In [1009]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import re

In [1010]:
device = ('cuda' if torch.cuda.is_available() else 'cpu')

In [1011]:
device

'cuda'

In [1012]:
train_dataset = pd.read_csv("train.csv", encoding="latin1")
test_dataset = pd.read_csv("test.csv", encoding="latin1")

In [1013]:
train_dataset.columns

Index(['textID', 'text', 'selected_text', 'sentiment', 'Time of Tweet',
       'Age of User', 'Country', 'Population -2020', 'Land Area (Km²)',
       'Density (P/Km²)'],
      dtype='object')

In [1014]:
train_dataset.head()

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26


In [1015]:
test_dataset.columns

Index(['textID', 'text', 'sentiment', 'Time of Tweet', 'Age of User',
       'Country', 'Population -2020', 'Land Area (Kmï¿½)',
       'Density (P/Kmï¿½)'],
      dtype='object')

In [1016]:
test_dataset.head()

,textID,text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Kmï¿½),Density (P/Kmï¿½)
0,f87dea47db,Last session of the day http://twitpic.com/67ezh,neutral,morning,0-20,Afghanistan,38928346,652860.0,60
1,96d74cb729,Shanghai is also really exciting (precisely -...,positive,noon,21-30,Albania,2877797,27400.0,105
2,eee518ae67,"Recession hit Veronique Branquinho, she has to...",negative,night,31-45,Algeria,43851044,2381740.0,18
3,01082688c6,happy bday!,positive,morning,46-60,Andorra,77265,470.0,164
4,33987a8ee5,http://twitpic.com/4w75p - I like it!!,positive,noon,60-70,Angola,32866272,1246700.0,26


In [1017]:
train_dataset = train_dataset.loc[:,["text", "sentiment"]]

In [1018]:
test_dataset = test_dataset.loc[:, ["text", "sentiment"]]

In [1019]:
train_dataset.head()

,text,sentiment
0,"I`d have responded, if I were going",neutral
1,Sooo SAD I will miss you here in San Diego!!!,negative
2,my boss is bullying me...,negative
3,what interview! leave me alone,negative
4,"Sons of ****, why couldn`t they put them on t...",negative


In [1020]:
test_dataset.head()

,text,sentiment
0,Last session of the day http://twitpic.com/67ezh,neutral
1,Shanghai is also really exciting (precisely -...,positive
2,"Recession hit Veronique Branquinho, she has to...",negative
3,happy bday!,positive
4,http://twitpic.com/4w75p - I like it!!,positive


In [1021]:
train_dataset.shape

(27481, 2)

In [1022]:
def tokenizer(text):
    text = str(text)
    lower_case_text = text.lower()
    # removes the characters that are not letter or digit or space with ""
    text = re.sub(r"[^\w\d\s]", "", lower_case_text)
    return text.split()

In [1023]:
tokenizer("Hello MY NaMe is AbhIHsheK TimMsInA")

['hello', 'my', 'name', 'is', 'abhihshek', 'timmsina']

In [1024]:
tokenizer(train_dataset.iloc[0,0])

['id', 'have', 'responded', 'if', 'i', 'were', 'going']

In [1025]:
answers = {}
choices = train_dataset.iloc[:, 1].unique()
print(choices)
for i in choices:
    answers[i] = len(answers)
answers

['neutral' 'negative' 'positive']


{'neutral': 0, 'negative': 1, 'positive': 2}

In [1026]:
# voccabulary for stroing the tokens
voccab = {
    '<UNK>' : 0
}

# for building the voccabulary for training as it is done in timestep, each token means one timestep
def build_voccab(dataset):
    for text_index in range(dataset.shape[0]):
        text = dataset.iloc[text_index,0]
        # tokenize
        tokens = tokenizer(text)
        # add in voccab
        for token in tokens:
            if token not in voccab:
                voccab[token] = len(voccab)
    print("voccab created ✅")

In [1027]:
build_voccab(train_dataset)

voccab created ✅


In [1028]:
len(voccab)

29584

In [1029]:
def text_to_indices(text, voccab):
    tokens = tokenizer(text)
    indices = []
    if len(tokens) == 0:
        indices.append(voccab['<UNK>'])
        return torch.tensor(indices)
    for token in tokens:
        if token in voccab:
            indices.append(voccab[token])
        else:
            indices.append(voccab["<UNK>"])
    return torch.tensor(indices)

In [1030]:
indices = text_to_indices("i am dsd", voccab)
indices

tensor([  5, 112,   0])

In [1031]:
len(voccab)

29584

In [1032]:
# location where the data are stored
class CustomDataSet(Dataset):
    def __init__(self, df, voccab):
        self.df = df
        self.voccab = voccab
        print(self.df.shape)
        print(len(self.voccab))

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, index):
        indices = text_to_indices(self.df.iloc[index, 0], self.voccab)
        answer = torch.tensor(answers[self.df.iloc[index, 1]])
        return indices, answer

In [1033]:
train_data = CustomDataSet(train_dataset, voccab)

(27481, 2)
29584


In [1034]:
train_data.__getitem__(0)

(tensor([1, 2, 3, 4, 5, 6, 7]), tensor(0))

In [1035]:
train_dataloader = DataLoader(dataset = train_data, batch_size = 64, shuffle = True)

In [1036]:
len(train_dataloader)

430

In [1037]:
class RNN(nn.Module):
    def __init__(self, size_of_voccab, size_of_embedding):

        super().__init__()
        
        # embedding so that the model can understand the tokens as the integer is nothing but id
        # we need a embedding that explains the meaning of the token to the rnn
        # as the model learns, the embedding also improves
        self.embedding = nn.Embedding(num_embeddings= size_of_voccab, embedding_dim= size_of_embedding)
        # RNN as the sequential model, that process the embedding of each token sequentially and understands/learns/summarizes the tokens one timestep at a time
        # it uses the backward loop to use previous hidden state in the hidden layer of next time step as context/memroy to process the nextt token
        # with such mechanism it keeps summarizing till the end and
        # it gives 64 output by taking embedding of each token
        self.rnn = nn.RNN(size_of_embedding, 64)
        # linear is used to combine the output into 3 neurons(positive, negative and neutral)
        self.linear = nn.Linear(64, 3)

    # Forward Propagation
    def forward(self, tokens_indices):
        embeddings = self.embedding(tokens_indices)
        rnn_output = self.rnn(embeddings)
        linear = self.linear(rnn_output[1])
        return linear

In [1038]:
rnn_model = RNN(len(voccab), 50)
rnn_model.to(device=device)

RNN(
  (embedding): Embedding(29584, 50)
  (rnn): RNN(50, 64)
  (linear): Linear(in_features=64, out_features=3, bias=True)
)

In [1039]:
train_data[0]

(tensor([1, 2, 3, 4, 5, 6, 7]), tensor(0))

In [1040]:
for i in rnn_model.named_parameters():
    print(i)

('embedding.weight', Parameter containing:
tensor([[ 0.1857, -0.0344,  0.5480,  ...,  0.6981,  0.0919, -0.5473],
        [-0.0798,  1.4558, -0.6159,  ..., -0.3911,  0.2495,  0.8338],
        [ 0.4508,  0.3039, -1.1223,  ...,  0.9322,  0.8765,  0.2675],
        ...,
        [-0.0925,  1.4526, -2.4394,  ...,  0.3007, -1.1327, -0.4694],
        [-0.4889, -0.6782,  1.0631,  ..., -0.4583,  1.1181, -2.2743],
        [-1.0142, -3.1066,  1.1393,  ..., -0.5630,  1.7875, -0.3079]],
       device='cuda:0', requires_grad=True))
('rnn.weight_ih_l0', Parameter containing:
tensor([[-0.1155,  0.1154, -0.0018,  ..., -0.0651, -0.0882, -0.1141],
        [-0.0824, -0.0810,  0.0552,  ..., -0.0027,  0.1023, -0.0136],
        [-0.0556, -0.0341,  0.0957,  ..., -0.0352,  0.0026,  0.0791],
        ...,
        [-0.1042,  0.0596, -0.0067,  ...,  0.0339,  0.1096,  0.0006],
        [-0.0132,  0.1207,  0.0952,  ...,  0.1088,  0.0843,  0.1248],
        [ 0.0481, -0.0528, -0.0866,  ..., -0.0558,  0.1250,  0.1032]],
 

In [1041]:
loss_fn = nn.CrossEntropyLoss()
optim = torch.optim.Adam(rnn_model.parameters(), lr = 0.001)

In [1042]:
epochs = 25

In [ ]:
# training the RNN model:
for epoch in range(epochs):
    tota_loss = 0
    for que, ans in train_dataloader:
        que = que.reshape(que.shape[1]).to(device)
        ans = ans.to(device)
        y_pred = rnn_model(que)
        optim.zero_grad()
        loss = loss_fn(y_pred, ans)
        loss.backward()
        optim.step()
        tota_loss = tota_loss + loss.item()
    print("Epoch :", epoch, "===============  Loss :", tota_loss/len(train_dataloader))

In [1043]:
tokens = tokenizer(test_dataset.iloc[4, 0])
tokens

['httptwitpiccom4w75p', 'i', 'like', 'it']

In [1044]:
index = text_to_indices(tokens, voccab).to(device=device)
index

tensor([  0,   5,  88, 133], device='cuda:0')

In [1045]:
rnn_model(index)

tensor([[0.0047, 0.0033, 0.1171]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [1046]:
test_dataset.shape

(3534, 2)

## testing using the test

In [1047]:
test_data = CustomDataSet(test_dataset, voccab)
test_dataloader = DataLoader(test_data, batch_size= 1, shuffle= False)

(3534, 2)
29584


In [1048]:
test_data.__len__()

3534

In [1049]:
test_data[1]

(tensor([    0,    19,   374,    87,  1476, 13469,     0, 16109,   454,  5523,
            14,  5780, 13200,     0]),
 tensor(2))

In [1050]:
rnn_model.eval()

RNN(
  (embedding): Embedding(29584, 50)
  (rnn): RNN(50, 64)
  (linear): Linear(in_features=64, out_features=3, bias=True)
)

In [ ]:
correct = 0
for question, answer in test_dataloader:
    question = question.reshape(question.shape[1]).to(device)
    answer = answer.to(device)
    y_pred = rnn_model(question)
    if answer == torch.max(y_pred, dim= 1)[1]:
        correct = correct + 1
    print("real :", answer)
    print(y_pred)
    print("pred :", torch.max(y_pred, dim = 1)[1])
    print("="*100)
print("Accuracy :", correct/len(test_dataloader))